# Part 5: Build an agent over the knowledge base

Part 5 at Caldova begins when the supply-chain team wants an agent to reason over evidence without asking the knowledge base to synthesize an answer. You will create an extractive knowledge base from the Part 1 knowledge sources, connect Microsoft Agent Framework to its MCP endpoint, then let the agent work with the retrieved evidence.

**🎯 Mission**
- Create a model-free knowledge base configured for extractive output
- Connect Microsoft Agent Framework to the knowledge base through its MCP endpoint
- Run an operational query and inspect the agent's tool activity

## Step 1: Set up the agent environment

Load the Foundry project, model deployment, tenant, and Foundry IQ settings from the project `.env` file. These settings create the extractive knowledge base and connect it to the Agent Framework model.

In [32]:
import os

from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.foundry import FoundryChatClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)

FOUNDRY_PROJECT_ENDPOINT = os.environ["MICROSOFT_FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]

KNOWLEDGE_BASE_NAME = "caldova-supply-chain-minimal-knowledge-base"
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_SEARCH_SCOPE = "https://search.azure.com/.default"

print("Agent environment loaded")

Agent environment loaded


## Step 2: Add Azure authentication to MCP requests

The knowledge base MCP endpoint accepts a Microsoft Entra bearer token. Create an Azure credential, then define a header provider that acquires a current Foundry IQ token for each MCP request.

In [25]:
credential = AzureDeveloperCliCredential(
    tenant_id=AZURE_TENANT_ID,
    process_timeout=60,
)

def get_mcp_headers(_):
    token = credential.get_token(AZURE_SEARCH_SCOPE)
    return {"Authorization": f"Bearer {token.token}"}

## Step 3: Create an extractive knowledge base

Create a new knowledge base over the three knowledge sources from Part 1. Minimal reasoning effort does not perform source selection, query planning, or iterative search. `EXTRACTIVE_DATA` returns retrieved evidence without attaching a model or synthesizing a knowledge base answer.

In [33]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

knowledge_source_names = [
    "file-docs-knowledge-source",
    "supplier-evidence-knowledge-source",
    "sourcing-documents-knowledge-source",
]

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY),
)
knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova extractive knowledge base for supply-chain evidence",
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in knowledge_source_names],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Extractive knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated")

Extractive knowledge base 'caldova-supply-chain-minimal-knowledge-base' created or updated


## Step 4: Create the knowledge base MCP tool

Every Foundry IQ knowledge base exposes a streamable HTTP MCP endpoint. Build the extractive knowledge base endpoint and expose only the `knowledge_base_retrieve` operation to the agent.

In [27]:
knowledge_base_endpoint = (
    f"{AZURE_SEARCH_SERVICE_ENDPOINT.rstrip('/')}"
    f"/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp"
    f"?api-version={AZURE_SEARCH_API_VERSION}"
)
knowledge_base_mcp_tool = MCPStreamableHTTPTool(
    name="caldova-knowledge-base",
    url=knowledge_base_endpoint,
    header_provider=get_mcp_headers,
    allowed_tools=["knowledge_base_retrieve"],
    load_prompts=False,
)

print(f"MCP tool connected to {knowledge_base_endpoint}")

MCP tool connected to https://ill344-search-k4crbjlin7fae.search.windows.net/knowledgebases/caldova-supply-chain-minimal-knowledge-base/mcp?api-version=2026-05-01-preview


## Step 5: Create the supply-chain agent

Create a code-first Agent Framework agent backed by the model deployed in the Foundry project. The knowledge base returns extracts without synthesis, while the agent reasons over that evidence to produce the final answer.

In [28]:
chat_client = FoundryChatClient(
    project_endpoint=FOUNDRY_PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=credential,
)
agent = Agent(
    client=chat_client,
    name="CaldovaSupplyChainAgent",
    instructions=(
        "You are a Caldova supply-chain assistant. "
        "Always use the Caldova knowledge base tool for supply-chain questions. "
        "Ground every answer in retrieved evidence, cite the source titles or identifiers, "
        "and keep policy, approved commitments, and operational evidence distinct. "
        "If the retrieved evidence is insufficient, say what information is unavailable."
    ),
    tools=[knowledge_base_mcp_tool],
    default_options={"store": False},
)

print(f"Agent '{agent.name}' created")

Agent 'CaldovaSupplyChainAgent' created


## Step 6: Run a query

Call `run()` with the temperature-excursion question from Part 1. The agent should invoke the extractive MCP retrieval tool, then reason over the returned evidence before explaining the event and distribution decision.

In [34]:
question = (
    "Shipment SHIP-VC-1026-BOS-FRA exceeded its temperature range. "
    "What happened, and can the 4,800 units be distributed?"
)
response = await agent.run(question)
display(Markdown(response.text))

Here’s what the evidence shows for **SHIP-VC-1026-BOS-FRA**.

**What happened**
- A **high-temperature excursion** occurred on **15 Oct 2026 from 08:30 to 11:00 UTC**.
- The shipment went above the validated **2.0–8.0 °C** range for **2 hr 30 min**, with a **maximum of 10.7 °C**.
- The event occurred in the **Frankfurt airport controlled-transfer area** during transfer from the aircraft container to refrigerated road transport.  
  Source: **EXC-VC-1026-04.pdf**, **TL-VCL-1026.pdf**

**Investigation finding**
- **Probable cause:** the active air container remained at the controlled-transfer bay **94 minutes beyond the validated handoff window** after a **ground-handler vehicle change**.
- **Contributing factor:** the replacement refrigerated vehicle arrived after the scheduled connection window.
- Packaging was reported **intact**, with **no opening or physical damage observed**.  
  Source: **EXC-VC-1026-04.pdf**

**Affected quantity and containment**
- The affected inventory was **12 insulated cartons / 4,800 units**, lot **CLD-VCL-1026-A**.
- Immediate actions taken:
  - moved back to **qualified 2–8 °C storage** at **11:06 UTC**
  - labeled **“QUARANTINE – DO NOT DISTRIBUTE”**
  - Caldova quality notified
  - logger and chain-of-custody records secured  
  Source: **EXC-VC-1026-04.pdf**

**Can the 4,800 units be distributed?**
- **Yes.** The final disposition says the product was **released for distribution**.
- The quality assessment under **stability case CAL-QA-2026-184** concluded that the exposure **remained within the approved stability allowance** for lot **CLD-VCL-1026-A**.
- **Quarantine was removed on 16 Oct 2026 at 18:10 UTC**, and the product **may proceed through the authorized distribution channel**.  
  Source: **EXC-VC-1026-04.pdf**

**Approvals**
- The disposition was signed by:
  - **Jonas Keller**, Valence Quality Operations
  - **Priya Desai**, Valence Responsible Person
  - **Mara Levin**, Caldova Director of Product Quality  
  Source: **EXC-VC-1026-04.pdf**

**Policy / approved commitment vs operational evidence**
- **Operational evidence:** excursion timing, peak temperature, location, quarantine, root cause, and release decision are documented in **EXC-VC-1026-04.pdf** and **TL-VCL-1026.pdf**.
- **Approved commitment:** the shipment was on an approved **Boston–Frankfurt 2–8 °C lane** under **LA-VCL-1026**; that authorization is referenced in **EXC-VC-1026-04.pdf** and **INV-SUP-008-2026-10.pdf**.
- I do **not** see additional product-specific distribution restrictions beyond the release decision in the retrieved records.

**Bottom line**
- The shipment exceeded range because of a **ground-handler transfer delay in Frankfurt** after a vehicle change.
- The **4,800 units were initially quarantined**, then **released after quality assessment**, so **they can be distributed through the authorized channel**.

## Step 7: Inspect agent activity

Agent responses include the messages produced during `run()`. Display each MCP tool call and its arguments to see the query the agent sent to the knowledge base before producing its final answer.

In [30]:
import json

table_rows = []
for message in response.messages:
    for content in message.contents:
        if content.type == "function_call":
            arguments = content.arguments
            if not isinstance(arguments, str):
                arguments = json.dumps(arguments, indent=2)
            arguments = arguments.replace("|", "\\|").replace("\n", "<br>")
            table_rows.append((content.name, arguments))

table = ["| Call | Tool | Arguments |", "| ---: | --- | --- |"]
for call_number, (tool_name, arguments) in enumerate(table_rows, start=1):
    table.append(f"| {call_number} | `{tool_name}` | `{arguments}` |")

display(Markdown("\n".join(table)))

| Call | Tool | Arguments |
| ---: | --- | --- |
| 1 | `knowledge_base_retrieve` | `{"queries":["SHIP-VC-1026-BOS-FRA temperature excursion disposition","SHIP-VC-1026-BOS-FRA deviation CAPA release 4800 units","SHIP-VC-1026-BOS-FRA stability excursion distribution decision"]}` |

## ✅ Mission complete

**What you built:**

* ✓ **Extractive knowledge base**: A model-free knowledge base returns evidence from the Part 1 sources without answer synthesis.
* ✓ **Authenticated MCP connection**: The agent reaches the extractive knowledge base with your Azure identity.
* ✓ **Agent Framework supply-chain agent**: A Foundry-backed agent reasons over the retrieved evidence to answer an operational question.

➡️ Continue to the [lab summary](../../instructions/summary.md).